# Семинар 9. Временные ряды

В этом семинаре мы разберем:
- EDA временных рядов: rolling statistics, seasonal decomposition, ACF/PACF, тесты стационарности
- Преобразования: log, differencing, deseasonalization
- Baseline модели: naive, seasonal naive, drift
- Экспоненциальное сглаживание: SES, Holt, Holt-Winters
- ARIMA / SARIMA: теория, подбор параметров, residual diagnostics
- ML-подход: lag features + CatBoost + TimeSeriesSplit
- Финальное сравнение всех методов

**Главная особенность временных рядов:** порядок наблюдений имеет значение. Многие привычные техники (случайный split, shuffle) ломают предположения модели.

In [ ]:
# Colab: install deps; locally use `uv run jupyter lab seminar.ipynb`
import sys
if "google.colab" in sys.modules:
    !pip install -q statsmodels catboost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing, Holt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

np.random.seed(42)

## 1. Данные

Используем два классических датасета:

- **AirPassengers** - ежемесячное количество пассажиров международных авиалиний, 1949-1960. Тренд вверх + **мультипликативная сезонность** (амплитуда растет с уровнем).
- **CO2** - концентрация CO2 в атмосфере, измерения обсерватории Мауна Лоа. Тренд вверх + **аддитивная сезонность** (постоянная амплитуда).

Оба ряда - отличные примеры для классической time series analysis.

In [ ]:
# AirPassengers из statsmodels
from statsmodels.datasets import get_rdataset, co2

air = get_rdataset('AirPassengers').data
# Преобразуем в правильный DatetimeIndex
air.columns = ['time', 'passengers']
# time - это decimal year (1949.0, 1949.083, ...)
# Преобразуем в даты
dates = pd.date_range(start='1949-01-01', periods=len(air), freq='MS')
air_ts = pd.Series(air['passengers'].values, index=dates, name='passengers')

# CO2 из statsmodels - уже с нормальным индексом
co2_data = co2.load_pandas().data
# Агрегируем до месячных
co2_ts = co2_data['co2'].resample('MS').mean().dropna()

print(f'AirPassengers: {len(air_ts)} observations, {air_ts.index.min()} to {air_ts.index.max()}')
print(f'CO2:           {len(co2_ts)} observations, {co2_ts.index.min()} to {co2_ts.index.max()}')
air_ts.head()

## 2. EDA

### 2.1 Визуализация ряда

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(air_ts, color='steelblue', linewidth=1.5)
axes[0].set_title('AirPassengers (multiplicative seasonality)', fontsize=13)
axes[0].set_ylabel('Passengers (thousands)')

axes[1].plot(co2_ts, color='darkgreen', linewidth=1.5)
axes[1].set_title('CO2 Mauna Loa (additive seasonality)', fontsize=13)
axes[1].set_ylabel('CO2 (ppm)')

plt.tight_layout()
plt.show()

Что видим:
- **AirPassengers**: очевидный восходящий тренд + сезонные циклы (летние пики). Амплитуда колебаний **растет** со временем - это мультипликативная сезонность.
- **CO2**: восходящий тренд + годовые циклы (растения поглощают CO2 летом, выделяют зимой). Амплитуда колебаний **постоянная** - аддитивная сезонность.

### 2.2 Rolling statistics

Rolling mean (скользящее среднее) сглаживает ряд и делает тренд более видимым. Rolling std показывает, как меняется волатильность во времени.

In [ ]:
window = 12  # годовое окно

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(air_ts, label='Original', color='steelblue', alpha=0.5)
ax.plot(air_ts.rolling(window).mean(), label=f'Rolling mean ({window})', color='red', linewidth=2)
ax.plot(air_ts.rolling(window).std(), label=f'Rolling std ({window})', color='orange', linewidth=2)

ax.set_title('AirPassengers: original vs rolling statistics', fontsize=13)
ax.set_ylabel('Passengers')
ax.legend(fontsize=11)
plt.show()

Rolling std (оранжевая линия) растёт - значит волатильность увеличивается. Это признак **нестационарности** и мультипликативной природы сезонности.

### 2.3 Seasonal decomposition

Разложение ряда на компоненты:
- **Trend**: долгосрочное направление
- **Seasonal**: циклический паттерн (с фиксированным периодом)
- **Residual**: то, что не объяснили trend и seasonal

Две модели:
- **Аддитивная**: $Y_t = T_t + S_t + R_t$ (постоянная амплитуда сезонности)
- **Мультипликативная**: $Y_t = T_t \cdot S_t \cdot R_t$ (амплитуда растет с уровнем)

In [ ]:
# Аддитивная декомпозиция AirPassengers - плохо работает
decomp_add = seasonal_decompose(air_ts, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10))
axes[0].plot(decomp_add.observed); axes[0].set_title('Observed')
axes[1].plot(decomp_add.trend, color='orange'); axes[1].set_title('Trend')
axes[2].plot(decomp_add.seasonal, color='green'); axes[2].set_title('Seasonal')
axes[3].plot(decomp_add.resid, color='red'); axes[3].set_title('Residual (НЕ случайный!)')
plt.suptitle('AirPassengers: additive decomposition (плохо подходит)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Мультипликативная декомпозиция AirPassengers - хорошо работает
decomp_mult = seasonal_decompose(air_ts, model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10))
axes[0].plot(decomp_mult.observed); axes[0].set_title('Observed')
axes[1].plot(decomp_mult.trend, color='orange'); axes[1].set_title('Trend')
axes[2].plot(decomp_mult.seasonal, color='green'); axes[2].set_title('Seasonal')
axes[3].plot(decomp_mult.resid, color='red'); axes[3].set_title('Residual (случайный)')
plt.suptitle('AirPassengers: multiplicative decomposition (правильный выбор)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Для CO2 - аддитивная декомпозиция подходит
decomp_co2 = seasonal_decompose(co2_ts, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10))
axes[0].plot(decomp_co2.observed, color='darkgreen'); axes[0].set_title('Observed')
axes[1].plot(decomp_co2.trend, color='orange'); axes[1].set_title('Trend')
axes[2].plot(decomp_co2.seasonal, color='blue'); axes[2].set_title('Seasonal')
axes[3].plot(decomp_co2.resid, color='red'); axes[3].set_title('Residual')
plt.suptitle('CO2: additive decomposition (правильный выбор)', fontsize=14)
plt.tight_layout()
plt.show()

**Правило выбора**: если амплитуда колебаний растет вместе с уровнем ряда - мультипликативная. Если постоянна - аддитивная. Визуальная проверка по исходному графику.

### 2.4 ACF и PACF

**ACF (Autocorrelation Function)** - корреляция ряда с его лагами (сдвинутыми версиями). $ACF(k) = corr(Y_t, Y_{t-k})$.

**PACF (Partial Autocorrelation Function)** - корреляция ряда с лагом $k$, с удалённым влиянием промежуточных лагов. $PACF(k)$ = "прямое" влияние $Y_{t-k}$ на $Y_t$.

Голубая зона - 95% доверительный интервал. Значения внутри неё - статистически не отличимы от нуля.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf(air_ts, lags=40, ax=axes[0, 0])
axes[0, 0].set_title('AirPassengers: ACF')
plot_pacf(air_ts, lags=40, ax=axes[0, 1], method='ywm')
axes[0, 1].set_title('AirPassengers: PACF')

plot_acf(co2_ts, lags=40, ax=axes[1, 0])
axes[1, 0].set_title('CO2: ACF')
plot_pacf(co2_ts, lags=40, ax=axes[1, 1], method='ywm')
axes[1, 1].set_title('CO2: PACF')

plt.tight_layout()
plt.show()

Что видим:
- **AirPassengers ACF**: медленно затухает - признак **нестационарности** (тренда). Пики на лагах 12, 24 - годовая сезонность.
- **AirPassengers PACF**: большой пик на лаге 1 (значение зависит от предыдущего), значимые пики на 12-13.
- **CO2 ACF/PACF**: похожие паттерны.

Правила чтения (классика Box-Jenkins):
- AR(p): PACF обрывается после лага $p$, ACF затухает
- MA(q): ACF обрывается после лага $q$, PACF затухает
- Нестационарный ряд: ACF медленно затухает → нужно дифференцирование

### 2.5 Тест стационарности (ADF)

**Стационарность** - свойство ряда, когда его статистические характеристики (среднее, дисперсия, автокорреляция) не меняются со временем. Большинство классических моделей требуют стационарности.

**Augmented Dickey-Fuller (ADF) test**:
- H0: ряд **нестационарный** (есть единичный корень)
- H1: ряд стационарный
- Отклоняем H0 если p-value < 0.05

In [ ]:
def check_stationarity(series, name=''):
    result = adfuller(series.dropna())
    print(f'--- {name} ---')
    print(f'ADF Statistic:    {result[0]:.4f}')
    print(f'p-value:          {result[1]:.4f}')
    print(f'Critical (5%):    {result[4]["5%"]:.4f}')
    verdict = 'СТАЦИОНАРНЫЙ' if result[1] < 0.05 else 'НЕСТАЦИОНАРНЫЙ'
    print(f'Verdict:          {verdict}\n')

check_stationarity(air_ts, 'AirPassengers (original)')
check_stationarity(co2_ts, 'CO2 (original)')

Оба ряда нестационарные (ожидаемо - у обоих явный тренд). Нужно преобразование.

## 3. Преобразования

Цель: привести ряд к стационарному виду для классических моделей.

### 3.1 Log transform

$log$ превращает мультипликативную сезонность в аддитивную. Полезно для рядов с растущей амплитудой.

In [ ]:
air_log = np.log(air_ts)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(air_ts, color='steelblue')
axes[0].set_title('AirPassengers (original) - амплитуда растёт')
axes[1].plot(air_log, color='purple')
axes[1].set_title('log(AirPassengers) - амплитуда стабильна')
plt.tight_layout()
plt.show()

### 3.2 Differencing (дифференцирование)

**First difference**: $\Delta Y_t = Y_t - Y_{t-1}$ - убирает линейный тренд.

**Seasonal difference**: $\Delta_s Y_t = Y_t - Y_{t-s}$ (например, $s=12$ для месячных данных с годовой сезонностью) - убирает сезонность.

In [ ]:
# Дифференцируем лог-ряд
air_log_diff = air_log.diff().dropna()                # first diff (тренд)
air_log_sdiff = air_log.diff(12).dropna()             # seasonal diff (сезонность)
air_log_both = air_log.diff().diff(12).dropna()       # обе

fig, axes = plt.subplots(4, 1, figsize=(14, 11))
axes[0].plot(air_log, color='purple'); axes[0].set_title('log(AirPassengers)')
axes[1].plot(air_log_diff, color='blue'); axes[1].set_title('Δ log (первая разность - убирает тренд)')
axes[2].plot(air_log_sdiff, color='green'); axes[2].set_title('Δ₁₂ log (сезонная разность - убирает сезонность)')
axes[3].plot(air_log_both, color='red'); axes[3].set_title('Δ Δ₁₂ log (обе - стационарный ряд)')
plt.tight_layout()
plt.show()

In [ ]:
# Проверим стационарность после преобразований
check_stationarity(air_log, 'log(AirPassengers)')
check_stationarity(air_log_diff, 'Δ log(AirPassengers)')
check_stationarity(air_log_both, 'Δ Δ₁₂ log(AirPassengers)')

In [ ]:
# ACF после преобразований
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(air_log_both, lags=40, ax=axes[0])
axes[0].set_title('ACF: Δ Δ₁₂ log(AirPassengers)')
plot_pacf(air_log_both, lags=40, ax=axes[1], method='ywm')
axes[1].set_title('PACF: Δ Δ₁₂ log(AirPassengers)')
plt.tight_layout()
plt.show()

Большинство лагов внутри доверительной зоны - остался слабый сигнал на лагах 1 и 12. Это кандидаты для AR/MA компонент SARIMA.

## 4. Baseline модели

Перед сложными моделями всегда строим простейшие baseline. Если сложная модель не превосходит naive - она бесполезна.

### 4.1 Разделение train/test (по времени!)

**Главное правило:** нельзя делать случайный split. Тест - всегда в конце.

In [ ]:
# Последние 24 месяца (2 года) - тест
test_size = 24
train = air_ts.iloc[:-test_size]
test = air_ts.iloc[-test_size:]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train, label='Train', color='steelblue')
ax.plot(test, label='Test', color='orange')
ax.axvline(x=test.index[0], color='red', linestyle='--', alpha=0.5, label='Split')
ax.set_title(f'AirPassengers: train ({len(train)}) / test ({len(test)})')
ax.legend()
plt.show()

### 4.2 Метрики

In [ ]:
def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {'name': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []

### 4.3 Naive baselines

- **Naive**: прогноз = последнее значение
- **Seasonal Naive**: прогноз = значение из прошлого сезона (год назад для monthly data)
- **Drift**: прогноз = последнее значение + средний прирост

In [ ]:
# Naive: последнее значение
pred_naive = pd.Series(train.iloc[-1], index=test.index)

# Seasonal Naive: значение год назад
pred_snaive = pd.Series(
    [train.iloc[-12 + i % 12] for i in range(len(test))],
    index=test.index,
)

# Drift
slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
pred_drift = pd.Series(
    [train.iloc[-1] + slope * (i + 1) for i in range(len(test))],
    index=test.index,
)

results.append(evaluate(test, pred_naive, 'Naive'))
results.append(evaluate(test, pred_snaive, 'Seasonal Naive'))
results.append(evaluate(test, pred_drift, 'Drift'))

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train[-36:], label='Train (last 3 years)', color='steelblue')
ax.plot(test, label='Test (actual)', color='black', linewidth=2)
ax.plot(pred_naive, label='Naive', linestyle='--', color='red')
ax.plot(pred_snaive, label='Seasonal Naive', linestyle='--', color='green')
ax.plot(pred_drift, label='Drift', linestyle='--', color='orange')
ax.set_title('Baseline forecasts')
ax.legend(fontsize=11)
plt.show()

pd.DataFrame(results).round(2)

Seasonal Naive сразу в разы лучше обычного Naive - ряд явно сезонный. Это наш главный baseline: сложные модели должны быть лучше него.

## 5. Экспоненциальное сглаживание

Идея: взвешенное среднее прошлых значений, где вес убывает экспоненциально с возрастом наблюдения.

### 5.1 Simple Exponential Smoothing (SES)

$\hat{y}_{t+1} = \alpha y_t + (1-\alpha) \hat{y}_t$

Параметр $\alpha \in [0, 1]$: $\alpha$ = 1 → только последнее значение, $\alpha \to 0$ → все веса одинаковы. Подходит только для рядов **без тренда и сезонности**.

In [ ]:
model_ses = SimpleExpSmoothing(train).fit()
pred_ses = model_ses.forecast(len(test))
results.append(evaluate(test, pred_ses, 'SES'))
print(f'α (optimized): {model_ses.params["smoothing_level"]:.3f}')

### 5.2 Holt's method

SES + модель тренда. Два параметра: $\alpha$ для уровня, $\beta$ для тренда.

In [ ]:
model_holt = Holt(train).fit()
pred_holt = model_holt.forecast(len(test))
results.append(evaluate(test, pred_holt, 'Holt'))

### 5.3 Holt-Winters

Holt + модель сезонности. Три параметра: $\alpha, \beta, \gamma$. Два варианта (additive/multiplicative) соответствуют типу сезонности.

In [ ]:
model_hw_add = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=12).fit()
pred_hw_add = model_hw_add.forecast(len(test))

model_hw_mult = ExponentialSmoothing(train, trend='add', seasonal='mul', seasonal_periods=12).fit()
pred_hw_mult = model_hw_mult.forecast(len(test))

results.append(evaluate(test, pred_hw_add, 'Holt-Winters (add)'))
results.append(evaluate(test, pred_hw_mult, 'Holt-Winters (mult)'))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train[-36:], label='Train', color='steelblue', alpha=0.7)
ax.plot(test, label='Test (actual)', color='black', linewidth=2.5)
ax.plot(pred_ses, label='SES', linestyle='--', alpha=0.7)
ax.plot(pred_holt, label='Holt', linestyle='--', alpha=0.7)
ax.plot(pred_hw_add, label='Holt-Winters (add)', linestyle='--', linewidth=2)
ax.plot(pred_hw_mult, label='Holt-Winters (mult)', linestyle='--', linewidth=2)
ax.set_title('Exponential smoothing methods')
ax.legend(fontsize=10)
plt.show()

pd.DataFrame(results).round(2)

Holt-Winters (multiplicative) - значительно лучше всех baselines. Модель ловит и тренд, и сезонность, и её растущую амплитуду.

## 6. ARIMA / SARIMA

### 6.1 Теория

**ARIMA(p, d, q)** - три компоненты:
- **AR(p)**: Autoregressive - текущее значение зависит от $p$ предыдущих
- **I(d)**: Integrated - $d$-кратное дифференцирование для стационарности
- **MA(q)**: Moving Average - текущее значение зависит от $q$ предыдущих ошибок

**SARIMA(p, d, q)(P, D, Q)s** - ARIMA + сезонные компоненты:
- $(P, D, Q)$ - сезонные AR, I, MA
- $s$ - период сезонности (12 для месячных данных)

### 6.2 Выбор параметров через ACF/PACF

Смотрим на ACF/PACF дифференцированного ряда ($\Delta \Delta_{12} \log Y$):
- Пик на лаге 1 в ACF → MA(1) компонента → $q=1$
- Пик на лаге 12 в ACF → сезонная MA → $Q=1$
- Уже дифференцировали один раз обычно и сезонно → $d=1, D=1$

Типичный выбор: $SARIMA(0, 1, 1)(0, 1, 1)_{12}$

In [ ]:
# SARIMA(0,1,1)(0,1,1,12) на лог-ряде
train_log = np.log(train)

model_sarima = SARIMAX(
    train_log,
    order=(0, 1, 1),
    seasonal_order=(0, 1, 1, 12),
).fit(disp=False)

print(model_sarima.summary().tables[1])

In [ ]:
# Forecast на лог-шкале + confidence interval
forecast_result = model_sarima.get_forecast(steps=len(test))
pred_sarima_log = forecast_result.predicted_mean
ci_log = forecast_result.conf_int(alpha=0.05)

# Обратное преобразование в исходную шкалу
pred_sarima = np.exp(pred_sarima_log)
ci_lower = np.exp(ci_log.iloc[:, 0])
ci_upper = np.exp(ci_log.iloc[:, 1])

results.append(evaluate(test, pred_sarima, 'SARIMA(0,1,1)(0,1,1,12)'))

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train[-48:], label='Train', color='steelblue', alpha=0.7)
ax.plot(test, label='Test (actual)', color='black', linewidth=2.5)
ax.plot(pred_sarima, label='SARIMA forecast', color='red', linewidth=2)
ax.fill_between(test.index, ci_lower, ci_upper, color='red', alpha=0.15, label='95% CI')
ax.set_title('SARIMA(0,1,1)(0,1,1,12) forecast с доверительным интервалом')
ax.legend(fontsize=11)
plt.show()

### 6.3 Residual diagnostics

Хорошая модель SARIMA должна дать **случайные остатки** (white noise):
- Нет систематических паттернов в остатках vs время
- Остатки нормально распределены
- Нет автокорреляции в остатках

У `statsmodels` есть готовая функция `plot_diagnostics()`.

In [ ]:
fig = model_sarima.plot_diagnostics(figsize=(14, 10))
plt.suptitle('SARIMA residual diagnostics', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

Что смотрим:
- **Standardized residual** (верх-лево): должен быть белым шумом (нет видимых паттернов)
- **Histogram + KDE** (верх-право): должен быть близок к нормальному
- **Normal Q-Q** (низ-лево): точки должны лежать на красной линии
- **Correlogram** (низ-право): все лаги должны быть внутри доверительного интервала

Если что-то выглядит плохо - модель недоспецифицирована, нужно пробовать другие $(p,d,q)(P,D,Q)$.

### 6.4 Grid search по параметрам

Вместо ручного подбора можно перебрать несколько вариантов и выбрать по AIC.

In [ ]:
grid_results = []
for p in [0, 1]:
    for q in [0, 1]:
        for P in [0, 1]:
            for Q in [0, 1]:
                try:
                    m = SARIMAX(train_log, order=(p, 1, q), seasonal_order=(P, 1, Q, 12)).fit(disp=False)
                    grid_results.append({
                        'order': f'({p},1,{q})({P},1,{Q},12)',
                        'AIC': m.aic,
                        'BIC': m.bic,
                    })
                except Exception:
                    continue

grid_df = pd.DataFrame(grid_results).sort_values('AIC').reset_index(drop=True)
grid_df

Минимальный AIC - лучшая модель. BIC - альтернативный критерий со штрафом за сложность. Часто ($0,1,1$)$(0,1,1)_{12}$ - как раз оптимум.

## 7. ML-подход (кратко)

Классика - мощно, но требует предположений о стационарности, сезонности и т.д. ML-подход проще:
1. Строим признаки из самого ряда (lag features, rolling stats, date features)
2. Обучаем обычный регрессор (CatBoost/XGBoost)
3. Прогноз - предсказание регрессора

**Главная сложность**: правильная валидация через `TimeSeriesSplit`.

### 7.1 TimeSeriesSplit визуализация

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

fig, ax = plt.subplots(figsize=(14, 4))
for i, (tr_idx, val_idx) in enumerate(tscv.split(train)):
    ax.scatter(tr_idx, [i + 1] * len(tr_idx), c='steelblue', marker='_', s=30, label='Train' if i == 0 else None)
    ax.scatter(val_idx, [i + 1] * len(val_idx), c='orange', marker='_', s=30, label='Validation' if i == 0 else None)
ax.set_xlabel('Time index')
ax.set_ylabel('Fold')
ax.set_title('TimeSeriesSplit: train растёт, validation всегда после train')
ax.legend(fontsize=11)
ax.set_yticks(range(1, 6))
plt.show()

### 7.2 Feature engineering

In [ ]:
def make_features(series):
    df = pd.DataFrame({'y': series})
    # Lag features
    for lag in [1, 2, 3, 6, 12, 24]:
        df[f'lag_{lag}'] = df['y'].shift(lag)
    # Rolling statistics (shift first to avoid data leakage!)
    for window in [3, 6, 12]:
        df[f'roll_mean_{window}'] = df['y'].shift(1).rolling(window).mean()
        df[f'roll_std_{window}'] = df['y'].shift(1).rolling(window).std()
    # Date features
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['quarter'] = df.index.quarter
    return df

df_ml = make_features(air_ts)
df_ml.head(15)

In [ ]:
# Split по времени
df_train_ml = df_ml.iloc[:-test_size].dropna()
df_test_ml = df_ml.iloc[-test_size:]

feature_cols = [c for c in df_ml.columns if c != 'y']
X_tr, y_tr = df_train_ml[feature_cols], df_train_ml['y']
X_te, y_te = df_test_ml[feature_cols], df_test_ml['y']

model_cb = CatBoostRegressor(iterations=300, depth=4, learning_rate=0.05, verbose=False, random_state=42)
model_cb.fit(X_tr, y_tr)
pred_cb = pd.Series(model_cb.predict(X_te), index=X_te.index)
results.append(evaluate(test, pred_cb, 'CatBoost (lag features)'))

# Feature importance
fi = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_cb.feature_importances_,
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi['feature'], fi['importance'], color='steelblue')
ax.set_title('CatBoost feature importance')
plt.tight_layout()
plt.show()

Как и ожидалось, `lag_12` - самый важный признак (год назад), за ним `lag_1` и месяц. Модель сама "нащупала" сезонность через признаки.

## 8. Итоговое сравнение

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(train[-36:], label='Train', color='gray', alpha=0.5)
ax.plot(test, label='Actual', color='black', linewidth=3)
ax.plot(pred_snaive, label='Seasonal Naive', linestyle=':', alpha=0.8)
ax.plot(pred_hw_mult, label='Holt-Winters (mult)', linestyle='--', linewidth=2)
ax.plot(pred_sarima, label='SARIMA', linestyle='--', linewidth=2)
ax.plot(pred_cb, label='CatBoost', linestyle='--', linewidth=2)
ax.set_title('Финальное сравнение методов на тестовой выборке', fontsize=14)
ax.legend(fontsize=11)
plt.show()

In [ ]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric, color in zip(axes, ['MAE', 'RMSE', 'MAPE'], ['steelblue', 'orange', 'green']):
    sorted_df = results_df.sort_values(metric)
    ax.barh(sorted_df['name'], sorted_df[metric], color=color)
    ax.set_title(f'{metric} (lower is better)')
    ax.set_xlabel(metric)
plt.tight_layout()
plt.show()

results_df.round(2)

## Итоги

### Ключевые выводы
- **Всегда делайте baseline** (хотя бы Seasonal Naive). Сложная модель должна его превосходить.
- **Train/test split только по времени**, никаких случайных split или shuffle.
- **Визуализация - главный инструмент**. Декомпозиция, ACF/PACF, residual diagnostics - скажут больше, чем любая метрика.

### Когда что использовать
- **Экспоненциальное сглаживание (Holt-Winters)**: быстро, интерпретируемо, отличный baseline для серий с явным трендом и сезонностью.
- **SARIMA**: более гибко чем Holt-Winters, даёт confidence intervals из коробки, требует понимания ACF/PACF.
- **ML-подход (CatBoost + lag features)**: проще подключить внешние признаки (weather, events, holidays), масштабируется на multiple series, менее интерпретируем.

### Что осталось за кадром
- Prophet (Facebook) - похож на Holt-Winters но с регуляризацией
- LSTM / Transformers для длинных последовательностей
- Multivariate time series (VAR, VECM)
- State-space models, Kalman filter
- Anomaly detection во временных рядах